In [6]:
!python dataclean.py

Rows loaded: 112
Rows saved       : 111
Rows with comment: 111
Output saved to  : Gamage_Recruiters_Cleaned_Data.xlsx


# **Import Libraries**

In [7]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

print('Libraries imported successfully!')

Libraries imported successfully!


# **Load Data and Filter to Comments Only**

In [8]:
INPUT_FILE  = 'Gamage_Recruiters_Cleaned_Data.xlsx'
OUTPUT_FILE = 'Gamage_Recruiters_Sentiment_Results.xlsx'

df_all = pd.read_excel(INPUT_FILE, sheet_name='Cleaned_Data')
print(f'Total rows loaded : {len(df_all)}')

# Keep ONLY rows that have actual comment text 

df = df_all[
    df_all['comment_cleaned'].notna() &
    (df_all['comment_cleaned'].str.strip() != '')
].copy().reset_index(drop=True)

print(f'Rows with comments: {len(df)}')
print()
print('All comments we will analyze:')
for _, row in df.iterrows():
    print(f'  {row["mention_id"]} | {row["post_type"]} | {row["comment_cleaned"]}')

Total rows loaded : 111
Rows with comments: 21

All comments we will analyze:
  LI-087 | Job Post | what is the industry
  LI-090 | Job Post | exciting opportunity, highly recommend, sharing to my network
  LI-091 | Job Post | great opportunity, amazing opportunity
  LI-092 | Job Post | thanks for sharing, poornima, thanks for sharing, poornima, great opportunity
  LI-093 | Job Post | i highly recommend, great opportunity, amazing opportunity
  LI-094 | Job Post | exciting opportunity
  LI-095 | Job Post | cfbr
  LI-096 | Job Post | thanks for sharing
  LI-097 | Marketing Post | exciting opportunity
  LI-098 | Marketing Post | reposted for reach, exciting opportunity
  LI-099 | Marketing Post | looks great, this is great
  LI-100 | Job Post | sharing to my network, great opportunity
  LI-101 | Marketing Post | love this,
  LI-102 | Job Post | great opportunity, reposted for reach
  LI-103 | Job Post | great opportunity, reposted for reach, sharing to my network, exciting opportunity
  

# **Flag Low Quality Comments**

In [10]:

LOW_QUALITY_PHRASES = [
    'cfbr',
    'reposted for reach',
    'sharing to my network',
    'sharing job for reach',
    'thanks for sharing',
]

def is_low_quality(text):
    """
    Returns True if the comment is just a generic engagement
    phrase with no real sentiment value.
    """
    if pd.isna(text): return True
    text_lower = str(text).lower().strip()
    # Check if the entire comment is made of low quality phrases
    for phrase in LOW_QUALITY_PHRASES:
        text_lower = text_lower.replace(phrase, '').strip().strip(',').strip()
    # If nothing meaningful remains after removing those phrases
    return len(text_lower) < 5

df['is_meaningful'] = ~df['comment_cleaned'].apply(is_low_quality)

meaningful_count    = df['is_meaningful'].sum()
low_quality_count   = (~df['is_meaningful']).sum()

print(f'Meaningful comments : {meaningful_count}')
print(f'Low quality comments: {low_quality_count}')
print()
print('Low quality comments flagged (generic engagement phrases):')
for _, row in df[~df['is_meaningful']].iterrows():
    print(f'  {row["mention_id"]} | {row["comment_cleaned"]}')

print()
print('NOTE: Low quality rows are kept but flagged.')

Meaningful comments : 19
Low quality comments: 2

Low quality comments flagged (generic engagement phrases):
  LI-095 | cfbr
  LI-096 | thanks for sharing

NOTE: Low quality rows are kept but flagged.


# **VADER Sentiment Analysis**

In [11]:
vader = SentimentIntensityAnalyzer()

def get_vader_scores(text):
    """
    VADER scores explanation:
      compound >= 0.05  → Positive
      compound <= -0.05 → Negative
      in between        → Neutral
    These thresholds are from the official VADER research paper.
    """
    if not isinstance(text, str) or text.strip() == '':
        return {'vader_compound': 0.0, 'vader_pos': 0.0,
                'vader_neu': 1.0,  'vader_neg': 0.0,
                'vader_label': 'Neutral'}

    scores   = vader.polarity_scores(text)
    compound = scores['compound']

    if compound >= 0.05:
        label = 'Positive'
    elif compound <= -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'

    return {
        'vader_compound' : round(compound, 4),
        'vader_pos'      : round(scores['pos'], 4),
        'vader_neu'      : round(scores['neu'], 4),
        'vader_neg'      : round(scores['neg'], 4),
        'vader_label'    : label
    }

vader_results = df['comment_cleaned'].apply(get_vader_scores)
vader_df      = pd.json_normalize(vader_results)
df            = pd.concat([df, vader_df], axis=1)

print('VADER Complete!')
print(df['vader_label'].value_counts())
df[['mention_id', 'comment_cleaned', 'vader_compound', 'vader_label']].head(8)

VADER Complete!
vader_label
Positive    19
Neutral      2
Name: count, dtype: int64


,mention_id,comment_cleaned,vader_compound,vader_label
0,LI-087,what is the industry,0.0000,Neutral
1,LI-090,"exciting opportunity, highly recommend, sharin...",0.8973,Positive
2,LI-091,"great opportunity, amazing opportunity",0.9260,Positive
3,LI-092,"thanks for sharing, poornima, thanks for shari...",0.9538,Positive
4,LI-093,"i highly recommend, great opportunity, amazing...",0.9504,Positive
5,LI-094,exciting opportunity,0.7184,Positive
6,LI-095,cfbr,0.0000,Neutral
7,LI-096,thanks for sharing,0.6908,Positive


# **TextBlob Sentiment Analysis**

In [12]:
def get_textblob_scores(text):
    """
    TextBlob polarity:
      > 0  → Positive
      < 0  → Negative
      = 0  → Neutral
    """
    if not isinstance(text, str) or text.strip() == '':
        return {'tb_polarity': 0.0,
                'tb_subjectivity': 0.0,
                'tb_label': 'Neutral'}

    blob     = TextBlob(text)
    polarity = blob.sentiment.polarity
    subj     = blob.sentiment.subjectivity

    if polarity > 0:   label = 'Positive'
    elif polarity < 0: label = 'Negative'
    else:              label = 'Neutral'

    return {
        'tb_polarity'    : round(polarity, 4),
        'tb_subjectivity': round(subj, 4),
        'tb_label'       : label
    }

tb_results = df['comment_cleaned'].apply(get_textblob_scores)
tb_df      = pd.json_normalize(tb_results)
df         = pd.concat([df, tb_df], axis=1)

print('TextBlob Complete!')
print(df['tb_label'].value_counts())
df[['mention_id', 'comment_cleaned', 'tb_polarity', 'tb_label']].head(8)

TextBlob Complete!
tb_label
Positive    19
Neutral      2
Name: count, dtype: int64


,mention_id,comment_cleaned,tb_polarity,tb_label
0,LI-087,what is the industry,0.00,Neutral
1,LI-090,"exciting opportunity, highly recommend, sharin...",0.23,Positive
2,LI-091,"great opportunity, amazing opportunity",0.70,Positive
3,LI-092,"thanks for sharing, poornima, thanks for shari...",0.40,Positive
4,LI-093,"i highly recommend, great opportunity, amazing...",0.52,Positive
5,LI-094,exciting opportunity,0.30,Positive
6,LI-095,cfbr,0.00,Neutral
7,LI-096,thanks for sharing,0.20,Positive


# **Final Sentiment Label**

In [13]:
def assign_final_sentiment(row):
    # If both agree → high confidence
    # If disagree   → trust VADER
    if row['vader_label'] == row['tb_label']:
        return row['vader_label'], 'High'
    else:
        return row['vader_label'], 'Low'

df[['final_sentiment', 'confidence']] = df.apply(
    assign_final_sentiment, axis=1, result_type='expand'
)

total = len(df)
print('=' * 45)
print('FINAL SENTIMENT DISTRIBUTION')
print('=' * 45)
counts = df['final_sentiment'].value_counts()
for label, count in counts.items():
    pct = round(count / total * 100, 1)
    bar = '█' * int(pct / 2)
    print(f'  {label:<10}: {count:>2} rows ({pct}%)  {bar}')

print(f'\n  Total analyzed : {total} rows')
print(f'  High confidence: {(df["confidence"] == "High").sum()}')
print(f'  Low confidence : {(df["confidence"] == "Low").sum()}')

FINAL SENTIMENT DISTRIBUTION
  Positive  : 19 rows (90.5%)  █████████████████████████████████████████████
  Neutral   :  2 rows (9.5%)  ████

  Total analyzed : 21 rows
  High confidence: 21
  Low confidence : 0


# **View All Results Row by Row**

In [14]:
# With only 21 rows, print every single result clearly
print('COMPLETE RESULTS — ALL COMMENTS')
print('=' * 75)

for _, row in df.iterrows():
    quality_flag = '' if row['is_meaningful'] else '  ⚠ LOW QUALITY'
    print(f'\n  ID          : {row["mention_id"]}{quality_flag}')
    print(f'  Post Type   : {row["post_type"]}')
    print(f'  Comment     : {row["comment_text"]}')
    print(f'  Cleaned     : {row["comment_cleaned"]}')
    print(f'  VADER       : {row["vader_compound"]:>7}  → {row["vader_label"]}')
    print(f'  TextBlob    : {row["tb_polarity"]:>7}  → {row["tb_label"]}')
    print(f'  FINAL       : {row["final_sentiment"]}  (Confidence: {row["confidence"]})')
    print(f'  {"─" * 65}')

COMPLETE RESULTS — ALL COMMENTS

  ID          : LI-087
  Post Type   : Job Post
  Comment     : What is the industry
  Cleaned     : what is the industry
  VADER       :     0.0  → Neutral
  TextBlob    :     0.0  → Neutral
  FINAL       : Neutral  (Confidence: High)
  ─────────────────────────────────────────────────────────────────

  ID          : LI-090
  Post Type   : Job Post
  Comment     : Exciting opportunity, Highly recommend, Sharing to my network
  Cleaned     : exciting opportunity, highly recommend, sharing to my network
  VADER       :  0.8973  → Positive
  TextBlob    :    0.23  → Positive
  FINAL       : Positive  (Confidence: High)
  ─────────────────────────────────────────────────────────────────

  ID          : LI-091
  Post Type   : Job Post
  Comment     : Great opportunity, Amazing opportunity
  Cleaned     : great opportunity, amazing opportunity
  VADER       :   0.926  → Positive
  TextBlob    :     0.7  → Positive
  FINAL       : Positive  (Confidence: Hig

# **Save Results**

In [15]:
output_cols = [
    'mention_id', 'platform', 'date', 'post_type',
    'comment_text', 'comment_cleaned', 'is_meaningful',
    'vader_compound', 'vader_pos', 'vader_neu',
    'vader_neg', 'vader_label',
    'tb_polarity', 'tb_subjectivity', 'tb_label',
    'final_sentiment', 'confidence',
    'likes', 'shares', 'num_comments'
]

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    # Sheet 1 — all 21 comment rows with sentiment scores
    df[output_cols].to_excel(
        writer, sheet_name='Sentiment_Results', index=False)
    # Sheet 2 — rows excluded (no comments) for reference
    df_excluded = df_all[
        df_all['comment_cleaned'].isna() |
        (df_all['comment_cleaned'].str.strip() == '')
    ]
    df_excluded.to_excel(
        writer, sheet_name='Excluded_No_Comment', index=False)

print('=' * 50)
print('STEP 5 COMPLETE')
print('=' * 50)
print(f'  Saved to      : {OUTPUT_FILE}')
print(f'  Analyzed rows : {len(df)}  (comments only)')
print(f'  Excluded rows : {len(df_excluded)}  (no comments)')
print()
counts = df['final_sentiment'].value_counts()
for label, count in counts.items():
    pct = round(count / len(df) * 100, 1)
    print(f'  {label:<10}: {count} ({pct}%)')
print()

STEP 5 COMPLETE
  Saved to      : Gamage_Recruiters_Sentiment_Results.xlsx
  Analyzed rows : 21  (comments only)
  Excluded rows : 90  (no comments)

  Positive  : 19 (90.5%)
  Neutral   : 2 (9.5%)

